# Importações 

In [18]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

caminho = "../dataset/INMET_NE_PE_A341_CARUARU_01-01-2025_A_31-12-2025_ADAPTADO.CSV"
df_adaptado = pd.read_csv(caminho, sep=",", encoding='iso-8859-1', decimal=".")

# Tratando Valores Duplicados

Os dados dos registros dessa base de dados foram obtidos por meio de sensores automáticos de precisão, que medem as horas no padrão UTC. A medição acontece toda hora e o ano é composto por 8760 horas, que é justamente a quantidade de registros do DF. Isso significa que o esperado é que nenhum registro esteja duplicado, porque se não o Data Set teria mais do que 8760 linhas.

In [14]:
df_adaptado.duplicated().sum()

np.int64(0)

Como o esperado, nenhum registro duplicado foi identificado pelo Pandas. Vamos seguir para a seleção das variáveis relevantas das irrelevantes. 

# Removendo variáveis

Na base de dados adaptada com a coluna 'CHUVA' temos várias variáveis, só que algumas delas não são relevantes para a classificação de 'SIM' ou 'NAO':
- 'Data': Coluna que informa a data que o registro foi aferido. A data em si não impacta na chuva, então decidimos remover essa coluna;
- 'Hora UTC': Coluna que informa a hora que o registro foi aferido. A hora em si não impacta na chuva, então decidimos remover essa coluna;
- 'Unnamed: 19': Coluna que é apenas um erro de formatação do próprio pandas. Todos os registros dela são nulos. Sua remoção causa nenhum impacto.


In [15]:
df_adaptado.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 20 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   Data                                                   8760 non-null   str    
 1   Hora UTC                                               8760 non-null   str    
 2   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  8756 non-null   float64
 3   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)        8756 non-null   float64
 4   PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)       8756 non-null   float64
 5   RADIACAO GLOBAL (Kj/m²)                                5474 non-null   float64
 6   TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)           8756 non-null   float64
 7   TEMPERATURA DO PONTO DE ORVALHO (°C)                   8756 non-null   float64
 8   TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)             8756

In [19]:
df_adaptado.drop(columns= ["Data", "Hora UTC", "Unnamed: 19"], inplace=True)

C:\Users\dougl\AppData\Local\Temp\ipykernel_18888\801666477.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_adaptado.drop(columns= ["Data", "Hora UTC", "Unnamed: 19"], inplace=True)


In [28]:
df_adaptado.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 17 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  8756 non-null   float64
 1   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)        8756 non-null   float64
 2   PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)       8756 non-null   float64
 3   RADIACAO GLOBAL (Kj/m²)                                8760 non-null   float64
 4   TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)           8756 non-null   float64
 5   TEMPERATURA DO PONTO DE ORVALHO (°C)                   8756 non-null   float64
 6   TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)             8756 non-null   float64
 7   TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)             8756 non-null   float64
 8   TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)       8756

Após a remoção dessas colunas podemos seguir com nosso pré-processamento. Percebemos vários valores nulos em 'RADIACAO GLOBAL (Kj/m²)', e isso será tratado posteriormente.

# Tratando Outliers

Como já foi dito, os dados dos registros dessa base de dados foram obtidos por meio de sensores automáticos de precisão, não por uma pessoa. Isso signica que não há falha humana e, portanto, todos os registros são totalmente verdadeiros e não há outliers.

Podemos seguir com o pré-processamento.

# Tratando Inconsistências

Inconsistências normalmente aparecem em dados categórigos preenchidos por humanos, só que nossa base de dados não foi preenchida por pessoas e quase todas as variáveis são numéricas, e não categóricas. Apenas a variável-alvo é categórica: 'CHUVA', e ela não possui inconsistências.

Podemos prosseguir pro de tratamento de valores nulos.

In [17]:
df_adaptado["CHUVA"].value_counts()

CHUVA
NAO    7440
SIM    1320
Name: count, dtype: int64

# Tratando valores Nulos

Anteriormente identificamos que a coluna 'RADIACAO GLOBAL (Kj/m²)' possui apenas 5474 valores preenchidos, o que corresponde a aproximadamente 62,5% de todos os dados dessa variável. A radiação está diretamente ligada com a evaporação da água e, posteriormente, a chuva. Não podemos descartar essa coluna.

In [20]:
df_adaptado["RADIACAO GLOBAL (Kj/m²)"].info()

<class 'pandas.Series'>
RangeIndex: 8760 entries, 0 to 8759
Series name: RADIACAO GLOBAL (Kj/m²)
Non-Null Count  Dtype  
--------------  -----  
5474 non-null   float64
dtypes: float64(1)
memory usage: 68.6 KB


Decidimos usar a técnica de preenchimento pela média nessa coluna, já que não há risco de inconsitências e nem outliers.

In [21]:
media = df_adaptado["RADIACAO GLOBAL (Kj/m²)"].mean()
df_adaptado["RADIACAO GLOBAL (Kj/m²)"] = df_adaptado["RADIACAO GLOBAL (Kj/m²)"].fillna(media)

In [20]:
df_adaptado["RADIACAO GLOBAL (Kj/m²)"].info()

<class 'pandas.Series'>
RangeIndex: 8760 entries, 0 to 8759
Series name: RADIACAO GLOBAL (Kj/m²)
Non-Null Count  Dtype  
--------------  -----  
8760 non-null   float64
dtypes: float64(1)
memory usage: 68.6 KB


Além disso, No restante das outras colunas tivemos 4 registros nulos em cada uma. Vamos usar a técnica de remoção de valores nulos nas demais colunas. Decidimos fazer isso porque 4 registros representam uma quantidade irrelevante de valores quando comparados com o total (8760).

In [22]:
df_adaptado.dropna(inplace=True)

C:\Users\dougl\AppData\Local\Temp\ipykernel_18888\4212996644.py:1: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_adaptado.dropna(inplace=True)


In [10]:
df_adaptado.shape

(8756, 17)

# Label Encoding

Decidimos utilizar a técnica de Label Encoding, pra transformar os valores 'SIM' e 'NAO' em números inteiros

In [23]:
encoder = LabelEncoder()
df_adaptado["CHUVA"] = encoder.fit_transform(
     df_adaptado["CHUVA"]
)

In [36]:
df_adaptado['CHUVA'].value_counts()

CHUVA
0    7440
1    1320
Name: count, dtype: int64

# Normalização

Vamos usar a técnica de normalização de dados, pra que nosso modelo Decision Tree não trabalhe com valores muito grandes

In [24]:
lista_variaveis = [
     "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
     "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
     "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
     "RADIACAO GLOBAL (Kj/m²)",
     "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
     "TEMPERATURA DO PONTO DE ORVALHO (°C)",
     "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
     "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
     "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
     "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
     "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
     "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
     "UMIDADE RELATIVA DO AR, HORARIA (%)",
     "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
     "VENTO, RAJADA MAXIMA (m/s)",
     "VENTO, VELOCIDADE HORARIA (m/s)",
     "CHUVA"
]


scaler = MinMaxScaler()
df_adaptado[lista_variaveis] = scaler.fit_transform(
     df_adaptado[lista_variaveis]
)

In [13]:
df_adaptado.head()

,"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",CHUVA
0,0.427419,0.404959,0.411290,0.261342,0.310127,0.744361,0.343373,0.344828,0.658730,0.810606,0.909091,0.863014,0.915493,0.254190,0.596154,0.394366,0.0
1,0.467742,0.446281,0.435484,0.261342,0.291139,0.706767,0.295181,0.317241,0.650794,0.780303,0.924242,0.904110,0.901408,0.231844,0.519231,0.380282,0.0
2,0.443548,0.446281,0.451613,0.261342,0.272152,0.714286,0.283133,0.303448,0.634921,0.780303,0.924242,0.904110,0.929577,0.298883,0.461538,0.464789,0.0
3,0.395161,0.421488,0.395161,0.261342,0.253165,0.706767,0.259036,0.282759,0.619048,0.772727,0.939394,0.931507,0.943662,0.287709,0.493590,0.380282,0.0
4,0.354839,0.371901,0.362903,0.261342,0.240506,0.691729,0.246988,0.268966,0.619048,0.765152,0.954545,0.945205,0.957746,0.315642,0.461538,0.338028,0.0


# Finalização & Exportação

Após toda atapa de pré-processamento temos uma nova base de dados adequada para trabalhar com os modelos de IA. Segue o resultado:

In [25]:
df_adaptado.head(10)

,"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",CHUVA
0,0.427419,0.404959,0.411290,0.261342,0.310127,0.744361,0.343373,0.344828,0.658730,0.810606,0.909091,0.863014,0.915493,0.254190,0.596154,0.394366,0.0
1,0.467742,0.446281,0.435484,0.261342,0.291139,0.706767,0.295181,0.317241,0.650794,0.780303,0.924242,0.904110,0.901408,0.231844,0.519231,0.380282,0.0
2,0.443548,0.446281,0.451613,0.261342,0.272152,0.714286,0.283133,0.303448,0.634921,0.780303,0.924242,0.904110,0.929577,0.298883,0.461538,0.464789,0.0
3,0.395161,0.421488,0.395161,0.261342,0.253165,0.706767,0.259036,0.282759,0.619048,0.772727,0.939394,0.931507,0.943662,0.287709,0.493590,0.380282,0.0
4,0.354839,0.371901,0.362903,0.261342,0.240506,0.691729,0.246988,0.268966,0.619048,0.765152,0.954545,0.945205,0.957746,0.315642,0.461538,0.338028,0.0
5,0.346774,0.330579,0.354839,0.261342,0.234177,0.691729,0.234940,0.262069,0.611111,0.757576,0.954545,0.958904,0.957746,0.273743,0.346154,0.211268,0.0
6,0.330645,0.322314,0.338710,0.261342,0.240506,0.699248,0.240964,0.262069,0.611111,0.765152,0.954545,0.958904,0.957746,0.223464,0.358974,0.239437,0.0
7,0.330645,0.314050,0.338710,0.261342,0.240506,0.684211,0.240964,0.262069,0.611111,0.750000,0.954545,0.945205,0.943662,0.237430,0.416667,0.197183,0.0
8,0.362903,0.338843,0.338710,0.261342,0.246835,0.691729,0.234940,0.255172,0.595238,0.742424,0.939394,0.945205,0.943662,0.164804,0.326923,0.197183,0.0
9,0.379032,0.363636,0.370968,0.016502,0.246835,0.706767,0.234940,0.262069,0.611111,0.757576,0.954545,0.945205,0.957746,0.209497,0.397436,0.267606,0.0


In [26]:
df_adaptado.info()

<class 'pandas.DataFrame'>
Index: 8756 entries, 0 to 8759
Data columns (total 17 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  8756 non-null   float64
 1   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)        8756 non-null   float64
 2   PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)       8756 non-null   float64
 3   RADIACAO GLOBAL (Kj/m²)                                8756 non-null   float64
 4   TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)           8756 non-null   float64
 5   TEMPERATURA DO PONTO DE ORVALHO (°C)                   8756 non-null   float64
 6   TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)             8756 non-null   float64
 7   TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)             8756 non-null   float64
 8   TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)       8756 non-

Agora vamos exportar esse DF pra se tornar uma base de dados pré-processada

In [27]:
df_adaptado.to_csv('INMET_NE_PE_A341_CARUARU_01-01-2025_A_31-12-2025_PROCESSADO.CSV', index=False, encoding='iso-8859-1')